# Your First PydanticAI Agent

**What you will build:** a solid grasp of the `Agent` object — the one thing you'll reuse in every remaining notebook of Block 1. System prompts (static and dynamic), running, and reading the result.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/11_first_pydanticai_agent.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## The Agent object

An `Agent` bundles a model, a system prompt, its tools, and (later) an output type. You create it once and reuse it. Think of it as a configured, callable assistant.

In [ ]:
agent = Agent(model, system_prompt="You are a concise travel assistant. Answer in two sentences max.")

result = await agent.run("I have one day in Lisbon. What should I not miss?")
print(result.output)

```{tip}
In a notebook, always use `await agent.run(...)`. PydanticAI's `agent.run_sync(...)` raises `This event loop is already running` inside Jupyter/Colab, because the notebook is already inside an event loop. (`run_sync` is for plain scripts.)
```

```{note}
In standalone Python scripts (outside notebooks), use `agent.run_sync(...)` instead — no `await` needed. The async `await agent.run(...)` is for notebooks and async frameworks like FastAPI (which you'll use in Block 3).
```

```{note}
The current PydanticAI docs increasingly use `instructions=` (and `@agent.instructions`) rather than `system_prompt=`. Both work; the difference matters only with `message_history` (1.4): `instructions` from earlier runs are **not** re-sent, while a `system_prompt` message rides along in the history. We use `system_prompt=` throughout for consistency — if the docs show `instructions=`, it's the same idea.
```

## The result object

`result.output` is the answer. The result also carries the full message history and usage — the same things you inspected by hand in 0.1, now on a typed object.

In [ ]:
print("output:", result.output)
print("usage: ", result.usage)         # tokens, as in 0.1
print("messages:", len(result.all_messages()), "messages in the run")

### Anatomy of the messages

Two message classes, a handful of part kinds — this small vocabulary is the debugging language for the whole block (1.9 lives on it):

In [ ]:
for m in result.all_messages():
    print(type(m).__name__)                      # ModelRequest | ModelResponse
    for p in m.parts:
        content = str(getattr(p, "content", ""))[:70]
        print(f"   {p.part_kind:15s} {content}")

A `ModelRequest` is what *you* send (parts: `system-prompt`, `user-prompt` — and `tool-return` once tools exist); a `ModelResponse` is what the model answers (parts: `text` — and `tool-call`). It's 0.1's message list with the roles promoted to types. Two methods to keep straight: `all_messages()` returns the whole run *including* the request that started it; `new_messages()` returns only what this run added — the one you'll feed back as history in 1.4.

## The knobs from 0.1: `model_settings`

`temperature` and `max_tokens` didn't disappear — they moved into `model_settings`, either on the agent (a default for every run) or per run (an override):

In [ ]:
from pydantic_ai.settings import ModelSettings

# On the agent: every run of this agent is deterministic and short.
extractor = Agent(model,
                  system_prompt="Extract the city named in the user's message. Reply with the city only.",
                  model_settings=ModelSettings(temperature=0, max_tokens=20))
print((await extractor.run("I'm flying to Porto on Friday!")).output)

# Per run: same agent, creative just this once.
r = await agent.run("Give me a colourful one-liner about Lisbon.",
                    model_settings=ModelSettings(temperature=1.0))
print(r.output)

Same rules as 0.1: `temperature=0` for anything structured or repeatable, higher only for deliberate variety — and `max_tokens` truncates rather than errors, so a mysteriously short output means checking the same `finish_reason` logic you learned there.

## Static vs dynamic system prompts

The string you pass is a **static** system prompt. When the instructions depend on something computed at runtime (the date, the user's name), use a `@agent.system_prompt` function — it runs on every call and its return value is added to the system prompt.

In [ ]:
from datetime import date

agent = Agent(model, system_prompt="You are a helpful assistant.")

@agent.system_prompt
def add_today() -> str:
    return f"Today's date is {date.today().isoformat()}."

result = await agent.run("What's today's date, and is it a weekday?")
print(result.output)

```{note}
This solves a real problem: a static prompt can't know today's date, but the model's training data is frozen in the past. A dynamic system prompt is your first tool for **context engineering** — putting the right facts in front of the model at the right time.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Map the anatomy.** Run the travel question again and, without running code, predict the exact sequence of `part_kind`s in `all_messages()`. Then print it. **Done when:** your prediction matches.
2. **Race the settings.** Create two agents from the same model — one with `temperature=0`, one with `temperature=1.2` — and run the same slogan prompt three times through each. **Done when:** you can tell which is which from the outputs alone.
3. **Add another dynamic fact.** Add a second `@agent.system_prompt` that injects a fake "user's city", ask a location-dependent question, and then check with `all_messages()` *where* your injected line ended up. **Done when:** you've found your text inside the `system-prompt` part.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — The sequence** for a no-tool run is always: `system-prompt`, `user-prompt` (one ModelRequest) then `text` (one ModelResponse). Four kinds exist in total — the other two (`tool-call`, `tool-return`) appear the moment you add tools in 1.2.

**2 — Race the settings:**

```python
det = Agent(model, system_prompt="You write slogans.", model_settings=ModelSettings(temperature=0))
wild = Agent(model, system_prompt="You write slogans.", model_settings=ModelSettings(temperature=1.2))
for _ in range(3):
    print("t=0  :", (await det.run("Slogan for a mountain-town bakery.")).output)
for _ in range(3):
    print("t=1.2:", (await wild.run("Slogan for a mountain-town bakery.")).output)
```

The `t=0` lines repeat (or nearly); the `t=1.2` lines wander.

**3 — Dynamic fact:**

```python
@agent.system_prompt
def add_city() -> str:
    return "The user is in Bilbao."

result = await agent.run("What's a good local dish to try tonight?")
for m in result.all_messages():
    for p in m.parts:
        if p.part_kind == "system-prompt":
            print(p.content)
```

Both decorated functions' outputs land in the system-prompt content — dynamic prompts *compose*. That's the mechanism 0.7's environment injection used, formalized.
::::

**What's next:** in **1.2** we give the agent **tools** and learn why the tool *description* is the single biggest lever on agent quality.